# Project 72 — Synthetic Data Generator for Classification
## Generate Labeled Examples for Text Classification

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Classification Task

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd, json

llm = ChatOllama(model="qwen3:8b", temperature=0.8)

task = "email intent classification"
labels = ["inquiry", "complaint", "feedback", "request", "spam"]

print(f"Task: {task}")
print(f"Labels: {labels}")

## Step 2 — Generate Synthetic Examples Per Label

In [ ]:
gen_prompt = ChatPromptTemplate.from_messages([
    ("system", """Generate 5 realistic email subject+body examples for the label: {label}

Context: Corporate email classification system.
Make each example distinct in topic and writing style.

Return a JSON array: [{{"subject": "...", "body": "...", "label": "{label}"}}]"""),
    ("human", "Generate 5 {label} emails:")
])
gen_chain = gen_prompt | llm | StrOutputParser()

all_examples = []
for label in labels:
    raw = gen_chain.invoke({"label": label})
    try:
        s = raw.find("["); e = raw.rfind("]") + 1
        examples = json.loads(raw[s:e]) if s >= 0 else []
    except Exception:
        examples = [{"subject": f"Example {label}", "body": f"Sample {label} email", "label": label}]
    all_examples.extend(examples)
    print(f"  {label}: generated {len(examples)} examples")

df = pd.DataFrame(all_examples)
print(f"\nTotal: {len(df)} examples across {df['label'].nunique()} labels")
print(df['label'].value_counts().to_string())

## Step 3 — Validate with Cross-Check

In [ ]:
verify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify this email into one of: {labels}. Return ONLY the label."),
    ("human", "Subject: {subject}\nBody: {body}")
])
verify_chain = verify_prompt | llm | StrOutputParser()

correct = 0
for _, row in df.iterrows():
    predicted = verify_chain.invoke({
        "labels": ", ".join(labels),
        "subject": row.get("subject", ""),
        "body": row.get("body", ""),
    }).strip().lower()
    if row["label"].lower() in predicted:
        correct += 1

accuracy = correct / len(df) if len(df) > 0 else 0
print(f"Cross-validation accuracy: {accuracy:.0%}")
print(f"Consistent examples: {correct}/{len(df)}")

# Save
df.to_csv("sample_data/synthetic_classification.csv", index=False)
print("\n✓ Saved to sample_data/synthetic_classification.csv")

## What You Learned
- **Synthetic data generation** with label-conditioned prompts
- **Cross-validation** using LLM to verify label consistency
- **Dataset export** for downstream classification training